Original

In [ ]:
import random
import networkx as nx
import numpy as np


# -------------------------------
# Compute 1-hop and 2-hop dictionaries
# -------------------------------
def compute_hop_dicts_orig(G):
    one_hop_dict = {}
    two_hop_dict = {}

    for u in G.nodes():
        one_hop = set(G.neighbors(u))
        one_hop_dict[u] = one_hop

        two_hop = set()
        for v in one_hop:
            two_hop.update(G.neighbors(v))

        two_hop -= one_hop
        two_hop.discard(u)

        two_hop_dict[u] = two_hop

    return one_hop_dict, two_hop_dict


# -------------------------------
# LIE (Paper-faithful)
# -------------------------------
def LIE_orig(seed_set, one_hop_dict, two_hop_dict, p=0.01):
    seed_set = set(seed_set)

    one_hop = set()
    two_hop = set()

    for u in seed_set:
        one_hop.update(one_hop_dict[u])
        two_hop.update(two_hop_dict[u])

    one_hop -= seed_set
    two_hop -= seed_set
    two_hop -= one_hop

    sigma0 = len(seed_set)

    sigma1 = 0
    for u in one_hop:
        prob = 1.0
        for v in one_hop_dict[u]:
            if v in seed_set:
                prob *= (1 - p)
        sigma1 += (1 - prob)

    avg_sigma1 = sigma1 / max(len(one_hop), 1)

    sum_pu_du = 0
    for u in two_hop:
        prob = 1.0
        active_neighbors = 0

        for v in one_hop_dict[u]:
            if v in one_hop:
                prob *= (1 - p)
                active_neighbors += 1

        p_u_star = 1 - prob
        d_u_star = active_neighbors

        sum_pu_du += p_u_star * d_u_star

    sigma2_tilde = avg_sigma1 * sum_pu_du

    return sigma0 + sigma1 + sigma2_tilde


# -------------------------------
# Initialization
# -------------------------------
def initialize(G, n, k):
    nodes = list(G.nodes())
    degree_sorted = sorted(nodes, key=lambda x: G.degree(x), reverse=True)

    population = []

    for _ in range(n):
        Xi = degree_sorted[:k].copy()

        for j in range(k):
            if random.random() > 0.5:
                candidate = random.choice(nodes)
                if candidate not in Xi:
                    Xi[j] = candidate

        population.append(Xi)

    return population


# -------------------------------
# Random Walk
# -------------------------------
def random_walk_orig(candidates, k):
    new_pos = []

    while len(new_pos) < k:
        temp = random.choice(candidates)
        if temp not in new_pos:
            new_pos.append(temp)

    return new_pos


# -------------------------------
# Local Search
# -------------------------------
def local_search(G, xi, xi_score, one_hop_dict, two_hop_dict, p=0.01):
    xi = xi.copy()
    best_score = xi_score

    xi_sorted = sorted(xi, key=lambda x: G.degree(x))

    for node in xi_sorted:
        if node not in xi:
            continue

        for neigh in G.neighbors(node):
            if neigh not in xi:

                temp = xi.copy()
                idx = temp.index(node)
                temp[idx] = neigh

                temp_score = LIE_orig(temp, one_hop_dict, two_hop_dict, p)

                if temp_score > best_score:
                    xi = temp
                    best_score = temp_score
                    break

    return xi, best_score


# -------------------------------
# Candidate Generation
# -------------------------------
def generate_candidates_orig(G, k, alpha=0.5, beta=3):
    ks = nx.core_number(G)

    contribution = {}

    for node in G.nodes():
        structural = 1 / (G.degree(node) + 1)
        contribution[node] = alpha * ks[node] + (1 - alpha) * structural

    sorted_nodes = sorted(contribution, key=contribution.get, reverse=True)

    return sorted_nodes[:beta * k]


# -------------------------------
# Main DCSA
# -------------------------------
def DCSA(G, k, n=30, gmax=100, AP=0.1, p=0.01, fl=2):

    one_hop_dict, two_hop_dict = compute_hop_dicts_orig(G)

    population = initialize(G, n, k)
    memory = [x.copy() for x in population]

    pop_fitness = [LIE_orig(xi, one_hop_dict, two_hop_dict, p) for xi in population]
    mem_fitness = pop_fitness.copy()

    convergence = []

    best_index = np.argmax(pop_fitness)

    global_best = population[best_index].copy()
    global_best_score = pop_fitness[best_index]

    candidates = generate_candidates_orig(G, k)

    for iter in range(gmax):

        for i in range(n):

            j = random.randint(0, n - 1)
            r_i = random.random()

            if r_i >= AP:

                new_pos = population[i].copy()

                for idx in range(k):

                    x_ij = population[i][idx]
                    intersection_val = 0 if x_ij in memory[j] else 1

                    h_val = 0 if (r_i * fl * intersection_val) < 1 else 1

                    if h_val == 1:
                        candidate = random.choice(candidates)

                        if candidate not in new_pos:
                            new_pos[idx] = candidate

            else:
                new_pos = random_walk_orig(candidates, k)

            new_score = LIE_orig(new_pos, one_hop_dict, two_hop_dict, p)

            if new_score > pop_fitness[i]:
                population[i] = new_pos
                pop_fitness[i] = new_score

        # Memory update
        for i in range(n):
            if pop_fitness[i] > mem_fitness[i]:
                memory[i] = population[i].copy()
                mem_fitness[i] = pop_fitness[i]

        # Global best
        best_index = np.argmax(pop_fitness)

        current_best = population[best_index].copy()
        current_score = pop_fitness[best_index]

        if current_score > global_best_score:
            global_best = current_best.copy()
            global_best_score = current_score

        # Local Search every 5 iterations
        if iter % 5 == 0 and iter != 0:
            improved, improved_score = local_search(
                G, current_best, current_score,
                one_hop_dict, two_hop_dict, p
            )

            if improved_score > global_best_score:
                global_best = improved.copy()
                global_best_score = improved_score

        convergence.append(global_best_score)

    return global_best, global_best_score, convergence

Improved DCSA

In [ ]:
import random
import networkx as nx
import numpy as np

# ============================================================
# LIE CACHE
# Stores previously evaluated seed sets to avoid
# redundant influence computations
# ============================================================
_lie_cache = {}

def clear_lie_cache():
    global _lie_cache
    _lie_cache = {}


# ============================================================
# IMPROVED LIE (Probabilistic 2-Hop Estimation)
# Computes:
#   sigma0 -> seed count
#   sigma1 -> first-hop probabilistic activation
#   sigma2 -> second-hop probabilistic activation
# ============================================================
def LIE_improved(G, seed_set, p=0.01):

    # Convert seed set to set for faster lookup
    seed_set = set(seed_set)

    # --------------------------------------------------------
    # Collect one-hop and two-hop neighbors
    # --------------------------------------------------------
    one_hop = set()
    two_hop = set()

    # Find one-hop neighbors of seed nodes
    for u in seed_set:
        one_hop.update(G.neighbors(u))

    # Remove seed nodes from one-hop set
    one_hop -= seed_set

    # Find two-hop neighbors
    for u in one_hop:
        two_hop.update(G.neighbors(u))

    # Remove already visited nodes
    two_hop -= seed_set
    two_hop -= one_hop

    # --------------------------------------------------------
    # sigma0 = number of seed nodes
    # --------------------------------------------------------
    sigma0 = len(seed_set)

    # --------------------------------------------------------
    # sigma1 = first-hop probabilistic activation
    # --------------------------------------------------------
    node_sigma1 = {}
    sigma1 = 0.0

    for u in one_hop:

        # Probability that node u is NOT activated
        prob_not_active = 1.0

        for v in G.neighbors(u):
            if v in seed_set:
                prob_not_active *= (1 - p)

        # Activation probability of node u
        node_sigma1[u] = 1 - prob_not_active

        sigma1 += node_sigma1[u]

    # --------------------------------------------------------
    # sigma2 = second-hop probabilistic activation
    # --------------------------------------------------------
    sigma2 = 0.0

    for v in two_hop:

        # Probability that node v is NOT activated
        prob_not_activated = 1.0

        for w in G.neighbors(v):

            # Activated one-hop neighbors influence v
            if w in one_hop:
                prob_not_activated *= (1 - node_sigma1[w] * p)

        sigma2 += (1 - prob_not_activated)

    # Total influence estimation
    return sigma0 + sigma1 + sigma2


# ============================================================
# CACHED LIE EVALUATION
# Reuses previously computed influence values
# ============================================================
def LIE_cached(G, seed_set, p):

    # frozenset used as immutable cache key
    key = frozenset(seed_set)

    # Compute only if not already cached
    if key not in _lie_cache:
        _lie_cache[key] = LIE_improved(G, seed_set, p)

    return _lie_cache[key]


# ============================================================
# POPULATION INITIALIZATION
# Hybrid initialization:
#   - high-degree nodes
#   - random diversification
# ============================================================
def initialize_population(G, n, k):

    nodes = list(G.nodes())

    # Sort nodes by descending degree
    degree_sorted = sorted(
        nodes,
        key=lambda x: G.degree(x),
        reverse=True
    )

    population = []

    for _ in range(n):

        # Start with top-k high degree nodes
        xi = degree_sorted[:k].copy()

        # Add randomness for diversification
        for j in range(k):

            if random.random() < 0.4:

                candidate = random.choice(nodes)

                if candidate not in xi:
                    xi[j] = candidate

        population.append(xi)

    return population


# ============================================================
# RANDOM WALK
# Generates random candidate seed set
# ============================================================
def random_walk(G, k):

    nodes = list(G.nodes())

    return random.sample(nodes, k)


# ============================================================
# LIGHTWEIGHT LOCAL SEARCH
# Evaluates only top-3 highest-degree neighbors
# ============================================================
def local_search_light(G, xi, p):

    xi = xi.copy()

    # Current solution score
    base_score = LIE_cached(G, xi, p)

    # Iterate through each seed node
    for idx, node in enumerate(xi):

        neighbors = list(G.neighbors(node))

        # Skip isolated nodes
        if not neighbors:
            continue

        # ----------------------------------------------------
        # Select top-3 highest-degree neighbors only
        # ----------------------------------------------------
        neighbors = sorted(
            neighbors,
            key=lambda x: G.degree(x),
            reverse=True
        )[:3]

        best_candidate = None
        best_score = base_score

        # ----------------------------------------------------
        # Evaluate neighbor replacements
        # ----------------------------------------------------
        for neigh in neighbors:

            # Avoid duplicate seed nodes
            if neigh in xi:
                continue

            temp = xi.copy()

            # Replace current seed
            temp[idx] = neigh

            # Evaluate influence score
            score = LIE_cached(G, temp, p)

            # Accept better solution
            if score > best_score:
                best_score = score
                best_candidate = neigh

        # ----------------------------------------------------
        # Update solution if improvement found
        # ----------------------------------------------------
        if best_candidate is not None:
            xi[idx] = best_candidate
            base_score = best_score

    return xi


# ============================================================
# MAIN OPTIMIZED DCSA
# ============================================================
def optimized_DCSA(
    G,
    k,
    n=30,
    gmax=100,
    AP=0.1,
    p=0.01,
    fl=2,
    stagnation_limit=10
):

    # Clear previous cache
    clear_lie_cache()

    # --------------------------------------------------------
    # Initialize population and memory
    # --------------------------------------------------------
    population = initialize_population(G, n, k)

    # Personal memory of each crow
    memory = [x.copy() for x in population]

    # Evaluate initial population fitness
    pop_fitness = [LIE_cached(G, xi, p) for xi in population]

    mem_fitness = pop_fitness.copy()

    # --------------------------------------------------------
    # Initialize global best solution
    # --------------------------------------------------------
    best_idx = np.argmax(pop_fitness)

    global_best = population[best_idx].copy()

    global_best_score = pop_fitness[best_idx]

    convergence = []

    # Stagnation counter
    stagnation_counter = 0

    # ========================================================
    # MAIN ITERATION LOOP
    # ========================================================
    for iteration in range(gmax):

        prev_best = global_best_score

        # ----------------------------------------------------
        # Adaptive Awareness Probability
        # Increases exploration during stagnation
        # ----------------------------------------------------
        current_AP = min(
            0.4,
            AP + 0.02 * stagnation_counter
        )

        # ====================================================
        # POPULATION UPDATE
        # ====================================================
        for i in range(n):

            # Randomly select another crow
            j = random.randint(0, n - 1)

            # ------------------------------------------------
            # Exploration / Exploitation Decision
            # ------------------------------------------------
            if random.random() >= current_AP:

                # Follow another crow
                new_pos = population[i].copy()

                for idx in range(k):

                    # Update only if seed differs
                    if population[i][idx] not in memory[j]:

                        # Flight condition
                        if random.random() * fl >= 1:

                            candidate = memory[j][idx]

                            # Avoid duplicate seeds
                            if candidate not in new_pos:
                                new_pos[idx] = candidate

            else:

                # Random exploration
                new_pos = random_walk(G, k)

            # ------------------------------------------------
            # Evaluate updated solution
            # ------------------------------------------------
            new_score = LIE_cached(G, new_pos, p)

            # Accept improved solution
            if new_score > pop_fitness[i]:

                population[i] = new_pos

                pop_fitness[i] = new_score

        # ====================================================
        # MEMORY UPDATE
        # ====================================================
        for i in range(n):

            # Update personal best memory
            if pop_fitness[i] > mem_fitness[i]:

                memory[i] = population[i].copy()

                mem_fitness[i] = pop_fitness[i]

        # ====================================================
        # GLOBAL BEST UPDATE
        # ====================================================
        best_idx = np.argmax(pop_fitness)

        if pop_fitness[best_idx] > global_best_score:

            global_best = population[best_idx].copy()

            global_best_score = pop_fitness[best_idx]

        # ====================================================
        # LIGHTWEIGHT LOCAL SEARCH
        # Applied every 5 iterations
        # ====================================================
        if iteration % 5 == 0 and iteration != 0:

            improved = local_search_light(
                G,
                global_best,
                p
            )

            improved_score = LIE_cached(
                G,
                improved,
                p
            )

            # Accept refined solution
            if improved_score > global_best_score:

                global_best = improved

                global_best_score = improved_score

        # ====================================================
        # STAGNATION HANDLING
        # ====================================================
        if global_best_score <= prev_best:

            stagnation_counter += 1

        else:

            stagnation_counter = 0

        # ----------------------------------------------------
        # Population restart if stagnation persists
        # ----------------------------------------------------
        if stagnation_counter >= stagnation_limit:

            # Restart worst-performing 10%
            worst_indices = np.argsort(
                pop_fitness
            )[:max(1, n // 10)]

            for idx in worst_indices:

                # Random reinitialization
                population[idx] = random_walk(G, k)

                # Recompute fitness
                pop_fitness[idx] = LIE_cached(
                    G,
                    population[idx],
                    p
                )

            # Reset stagnation counter
            stagnation_counter = 0

        # Store convergence history
        convergence.append(global_best_score)

    # ========================================================
    # RETURN FINAL BEST SOLUTION
    # ========================================================
    return global_best, global_best_score, convergence

CELF

In [ ]:
import heapq
import time
import random

# ============================================================
# INDEPENDENT CASCADE MODEL
# ============================================================
def IC_model(G, seed_set, p=0.01, mc=100):
    total_spread = 0

    for _ in range(mc):

        active = set(seed_set)
        newly_active = set(seed_set)

        while newly_active:

            next_active = set()

            for u in newly_active:

                for v in G.neighbors(u):

                    if v not in active and random.random() < p:
                        next_active.add(v)

            newly_active = next_active
            active.update(newly_active)

        total_spread += len(active)

    return total_spread / mc


# ============================================================
# CELF ALGORITHM
# ============================================================
def CELF(
    G,
    k,
    p=0.01,
    mc=100
):

    start_time = time.time()

    queue = []

    # --------------------------------------------------------
    # Initial Marginal Gain Computation
    # --------------------------------------------------------
    for node in G.nodes():

        mg = IC_model(G, [node], p, mc)

        heapq.heappush(queue, (-mg, node, 0))

    seed_set = []
    spread = 0
    lookups = 0

    # --------------------------------------------------------
    # Main CELF Loop
    # --------------------------------------------------------
    while len(seed_set) < k:

        neg_mg, node, last_updated = heapq.heappop(queue)

        if last_updated == len(seed_set):

            seed_set.append(node)
            spread = IC_model(G, seed_set, p, mc)

        else:

            new_mg = (
                IC_model(G, seed_set + [node], p, mc)
                - IC_model(G, seed_set, p, mc)
            )

            heapq.heappush(
                queue,
                (-new_mg, node, len(seed_set))
            )

        lookups += 1

    runtime = time.time() - start_time

    return {
        "seeds": seed_set,
        "spread": spread,
        "runtime": runtime,
        "lookups": lookups
    }

DPSO

In [ ]:
import random
import numpy as np

# ============================================================
# DISCRETE PARTICLE SWARM OPTIMIZATION (DPSO)
# ============================================================
def DPSO_IM(
    G,
    k,
    n_particles=30,
    gmax=100,
    p=0.01,
    w=0.7,
    c1=1.5,
    c2=1.5
):

    clear_lie_cache()

    nodes = list(G.nodes())

    # --------------------------------------------------------
    # INITIALIZE PARTICLES
    # --------------------------------------------------------
    particles = [
        random.sample(nodes, k)
        for _ in range(n_particles)
    ]

    pbest = [x.copy() for x in particles]
    pbest_scores = [LIE_cached(G, x, p) for x in particles]

    best_idx = np.argmax(pbest_scores)
    gbest = pbest[best_idx].copy()
    gbest_score = pbest_scores[best_idx]

    convergence = []

    # --------------------------------------------------------
    # MAIN LOOP
    # --------------------------------------------------------
    for iteration in range(gmax):

        for i in range(n_particles):

            new_particle = particles[i].copy()

            for d in range(k):

                r = random.random()

                # Follow personal best
                if r < c1 / (c1 + c2):
                    candidate = pbest[i][d]

                # Follow global best
                else:
                    candidate = gbest[d]

                # Random exploration
                if random.random() < w:
                    candidate = random.choice(nodes)

                if candidate not in new_particle:
                    new_particle[d] = candidate

            new_score = LIE_cached(G, new_particle, p)

            particles[i] = new_particle

            if new_score > pbest_scores[i]:
                pbest[i] = new_particle.copy()
                pbest_scores[i] = new_score

        # ----------------------------------------------------
        # GLOBAL BEST UPDATE
        # ----------------------------------------------------
        best_idx = np.argmax(pbest_scores)

        if pbest_scores[best_idx] > gbest_score:
            gbest = pbest[best_idx].copy()
            gbest_score = pbest_scores[best_idx]

        convergence.append(gbest_score)

    return gbest, gbest_score, convergence

IC Model

In [ ]:
import random

def IC_model(G, seed_set, p=0.01, mc=10000):
    spread = 0

    for _ in range(mc):
        active = set(seed_set)
        new_active = set(seed_set)

        while new_active:
            temp = set()
            for u in new_active:
                for v in G.neighbors(u):
                    if v not in active:
                        if random.random() < p:
                            temp.add(v)
            new_active = temp
            active |= temp

        spread += len(active)

    return spread / mc

Graph load

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import time

# ============================================================
# DATASET PATHS
# ============================================================
dataset_files = {
    "HepTh": "/content/CA-HepTh.txt",
    "CaAstroPh": "/content/Ca-AstroPh.txt",
    "CondMat": "/content/CA-CondMat.txt"
}

# ============================================================
# GRAPH LOADER
# ============================================================
def load_graph(path):
    G = nx.read_edgelist(path, nodetype=int)
    return G.to_undirected()

# ============================================================
# PARAMETERS
# ============================================================
k = 10
p = 0.01
mc = 100

results = []

# ============================================================
# BENCHMARK LOOP
# ============================================================
for dataset_name, path in dataset_files.items():

    print(f"\n========== Running on {dataset_name} ==========")

    G = load_graph(path)

    # --------------------------------------------------------
    # Original DCSA
    # --------------------------------------------------------
    start = time.time()
    seed, score, conv = DCSA(G, k=k)
    runtime = time.time() - start
    spread = IC_model(G, seed, p, mc)

    results.append([
        dataset_name,
        "Original DCSA",
        spread,
        runtime,
        seed
    ])

    # --------------------------------------------------------
    # Improved DCSA
    # --------------------------------------------------------
    start = time.time()
    seed, score, conv = optimized_DCSA(G, k=k)
    runtime = time.time() - start
    spread = IC_model(G, seed, p, mc)

    results.append([
        dataset_name,
        "Improved DCSA",
        spread,
        runtime,
        seed
    ])

    # --------------------------------------------------------
    # CELF
    # --------------------------------------------------------
    celf_result = CELF(G, k=k, p=p, mc=mc)

    results.append([
        dataset_name,
        "CELF",
        celf_result["spread"],
        celf_result["runtime"],
        celf_result["seeds"]
    ])

    # --------------------------------------------------------
    # DPSO
    # --------------------------------------------------------
    start = time.time()
    seed, score, conv = DPSO_IM(G, k=k)
    runtime = time.time() - start
    spread = IC_model(G, seed, p, mc)

    results.append([
        dataset_name,
        "DPSO",
        spread,
        runtime,
        seed
    ])

# ============================================================
# RESULTS TABLE
# ============================================================
df = pd.DataFrame(
    results,
    columns=[
        "Dataset",
        "Algorithm",
        "Influence Spread",
        "Runtime (s)",
        "Seed Set"
    ]
)

df.to_csv("benchmark_results.csv", index=False)

print("\nBenchmark Completed.")
display(df)


========== Running on HepTh ==========


FileNotFoundError: [Errno 2] No such file or directory: '/content/HepTh.txt'

Comparision

In [ ]:
plt.figure(figsize=(10,6))

for algo in df["Algorithm"].unique():
    subset = df[df["Algorithm"] == algo]
    plt.plot(
        subset["Dataset"],
        subset["Influence Spread"],
        marker='o',
        label=algo
    )

plt.title("Influence Spread Comparison")
plt.xlabel("Dataset")
plt.ylabel("Influence Spread")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

for algo in df["Algorithm"].unique():
    subset = df[df["Algorithm"] == algo]
    plt.plot(
        subset["Dataset"],
        subset["Runtime (s)"],
        marker='o',
        label=algo
    )

plt.title("Runtime Comparison")
plt.xlabel("Dataset")
plt.ylabel("Runtime (seconds)")
plt.legend()
plt.grid(True)
plt.show()